In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

In [ ]:
stroke_data = pd.read_csv("stroke-prediction.csv")
X = stroke_data.drop(columns = ["stroke", "id"])
y = stroke_data["stroke"]
names = stroke_data.columns
print(names)

In [ ]:
features = ["age", "avg_glucose_level"]

X2 = StandardScaler().fit_transform(X[features])
Sigma = np.cov(X2, rowvar = False)

print("Covariance matrix:\n ", Sigma.round(2))

fig, ax = plt.subplots(figsize = (4, 3.6))
ax.scatter(X2[:, 0], X2[:, 1], s = 6, alpha = 0.5)
ax.set_xlabel("age (standardised)")
ax.set_ylabel("average glucose level (standardised)")
_ = ax.set_aspect("equal")

In [ ]:
thetas = np.linspace(0, np.pi, 180)
var_z = [np.var(X2 @ np.array([np.cos(t), np.sin(t)]), ddof = 1) for t in thetas]
best = thetas[np.argmax(var_z)]
worst = thetas[np.argmin(var_z)]

print(f"Principle axes 1: {best:.2f}")
print(f"Principal axes 2: {worst:.2f}")
print(f"Therefore, total variance is: {best + worst:.2f}")

fig, axs = plt.subplots(1, 2, figsize = (6, 2.6))
axs[0].plot(np.degrees(thetas), var_z)
axs[0].axvline(np.degrees(best), color = "k", linestyle = ":")
axs[0].axvline(np.degrees(worst), color = "k", linestyle = ":")
axs[0].set_xlabel(r"angle $\theta$ (degrees)")
axs[0].set_ylabel("variance of projection")
axs[1].scatter(X2[:, 0], X2[:, 1], s = 3, alpha = 0.4)
for t, c in [(best, "C1"), (best + np.pi / 2, "C2")]:
    axs[1].plot([-3 * np.cos(t), 3 * np.cos(t)], [-3 * np.sin(t), 3 * np.sin(t)], c)
axs[1].set_aspect("equal")
plt.tight_layout()

In [ ]:
evals, evecs = np.linalg.eigh(Sigma)
order = np.argsort(evals)[::-1]
evals, evecs = evals[order], evecs[:, order]
print("eigenvalues", evals.round(2), " sum:", evals.sum().round(2))
print("eigenvectors (columns):\n", evecs.round(3))
Z2 = X2 @ evecs
print("covariance of the scores:\n", np.cov(Z2, rowvar = False).round(2))

In [ ]:
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline

In [ ]:
# Convert categorical columns to numeric dummy variables
Xw = pd.get_dummies(X, drop_first=False, dtype=float)

# Convert any missing values, such as N/A in bmi, to the column median
Xw = Xw.apply(pd.to_numeric, errors="coerce")
Xw = Xw.fillna(Xw.median())

feature_names = Xw.columns.to_numpy()

raw = PCA().fit(Xw)
std = make_pipeline(StandardScaler(), PCA()).fit(Xw)

print("explained variance ratio, raw:",
      raw.explained_variance_ratio_[:4].round(3))
print("explained variance ratio, scaled:",
      std[-1].explained_variance_ratio_[:4].round(3))

j = np.argmax(np.abs(raw.components_[0]))

print(f"raw PC1 is dominated by '{feature_names[j]}'"
      f"(variance {Xw.iloc[:, j].var():.0f}; "
      f"next largest {np.sort(Xw.var(axis=0))[-2]:.0f})")

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression

cv = StratifiedKFold(5, shuffle=True, random_state = 0)

Zw = std.transform(Xw)

views = [("average glucose level vs age", Xw.iloc[:, :2].to_numpy()), 
        ("PCA, not scaled", raw.transform(Xw)[:, :2]), 
        ("PCA, standardized", Zw[:, :2])]

fig, axs = plt.subplots(1, 3, figsize = (18, 6))
for ax, (title, P2) in zip(axs, views):
    acc = cross_val_score(make_pipeline(StandardScaler(), LogisticRegression()), P2, y, cv = cv).mean()
    for c in range(3):
        ax.scatter(P2[y == c, 0], P2[y == c, 1], s = 6, label = f"class {c}")
    ax.set_title(f"{title}\nCV accuracy {acc:.2f}", fontsize = 9)
    ax.set_xticks([]); ax.set_yticks([])
_ = axs[0].legend(fontsize = 7)

## IGNORE EVERYTHING ABOVE THIS

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ai4_tools as ai4

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, validation_curve
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn import svm

MY_FILE = "stroke-prediction.csv"
raw = pd.read_csv(MY_FILE)

TARGET = "stroke"
DROP = ["id"]

df = raw.drop(columns=DROP)
y = df[TARGET]
X = df.drop(columns=[TARGET])

numeric_cols = X.select_dtypes("number").columns.tolist()
text_cols = [c for c in X.columns if c not in numeric_cols]

preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric_cols),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore"))]), text_cols),
])

C = 1.0  # SVM regularization parameter
models = {
    "SVC linear": svm.SVC(kernel="linear", C=C, class_weight = 'balanced'),
    "LinearSVC": svm.LinearSVC(C=C, max_iter=10000, class_weight = 'balanced'),
    "SVC rbf": svm.SVC(kernel="rbf", gamma=0.7, C=C, class_weight = 'balanced'),
    "SVC poly": svm.SVC(kernel="poly", degree=3, gamma="auto", C=C, class_weight = 'balanced'),
}

Xm_train, X_test, ym_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
cv = StratifiedKFold(5, shuffle=True, random_state=0)

def compare(models, X, y, cv, scoring=("roc_auc", "recall")):
    rows = []
    for name, model in models.items():
        pipe = Pipeline([("prep", preprocess), ("model", model)])
        s = cross_validate(pipe, X, y, cv=cv, scoring=list(scoring), return_train_score=True)
        row = {"model": name}
        for m in scoring:
            row[m] = s["test_" + m].mean()
            row[m + " spread"] = s["test_" + m].std()
        row["training " + scoring[0]] = s["train_" + scoring[0]].mean()
        row["fit ms"] = 1000 * s["fit_time"].mean()
        row["score ms per 1000 rows"] = 1000 * s["score_time"].mean() / (len(X) / cv.get_n_splits()) * 1000
        rows.append(row)
    return pd.DataFrame(rows).set_index("model").round(3)

results = compare(models, Xm_train, ym_train, cv)
results

,roc_auc,roc_auc spread,recall,recall spread,training roc_auc,fit ms,score ms per 1000 rows
model,,,,,,,
SVC linear,0.843,0.032,0.804,0.094,0.854,237.652,72.533
LinearSVC,0.845,0.029,0.824,0.103,0.856,20.700,20.893
SVC rbf,0.704,0.036,0.176,0.064,0.990,187.542,216.501
SVC poly,0.808,0.029,0.643,0.081,0.891,177.140,83.688


In [2]:
ai4.decision("AI4I: ",
             "",
             cost="")
ai4.decision_log()

Decision 1
  Chose:   AI4I: 
  Because: 
  Cost:    
  Runtime: Windows | AMD64 Family 23 Model 96 Stepping 1, AuthenticAMD (6 cores, 7.4 GB) | CPU only | Python 3.13.14


,choice,why,cost,runtime
0,AI4I:,,,"Windows | AMD64 Family 23 Model 96 Stepping 1,..."


#### Searching for dial, starting with C

In [3]:
from sklearn.model_selection import GridSearchCV

METRIC = "roc_auc"
TUNE = "LinearSVC"
GRID = {"model__C": np.logspace(-4, 3, 8)}

pipe = Pipeline([("prep", preprocess), ("model", models[TUNE])])

search = GridSearchCV(pipe, GRID, cv=cv, scoring=METRIC, return_train_score=True)
search.fit(Xm_train, ym_train)

grid_table = pd.DataFrame(search.cv_results_)[["params", "mean_train_score", "mean_test_score", "std_test_score"]]
print("best:", search.best_params_, " cross-validated {}: {:.4f}".format(METRIC, search.best_score_))

models[TUNE + ", tuned"] = search.best_estimator_.named_steps["model"]   # model only; compare() adds prep
grid_table.round(4)

best: {'model__C': np.float64(0.001)}  cross-validated roc_auc: 0.8460


,params,mean_train_score,mean_test_score,std_test_score
0,{'model__C': 0.0001},0.8429,0.8376,0.0259
1,{'model__C': 0.001},0.8549,0.8460,0.0280
2,{'model__C': 0.01},0.8558,0.8459,0.0296
3,{'model__C': 0.1},0.8560,0.8447,0.0291
4,{'model__C': 1.0},0.8560,0.8446,0.0290
5,{'model__C': 10.0},0.8560,0.8446,0.0290
6,{'model__C': 100.0},0.8560,0.8446,0.0290
7,{'model__C': 1000.0},0.8560,0.8446,0.0290


only a 0.001 improvement on roc_auc, so lets change gamma

In [4]:
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import RobustScaler, MinMaxScaler

METRIC = "recall"                       # used to pick the winner; switch to "recall" if that's your priority
OTHER = "roc_auc" if METRIC == "recall" else "recall"
SCORING = {"roc_auc": "roc_auc", "recall": "recall"}

def tune(name, grid):
    """Grid-search one model from `models` inside the full pipeline, using CV on the training split."""
    pipe = Pipeline([("prep", preprocess), ("model", models[name])])
    search = GridSearchCV(pipe, grid, cv=cv, scoring=SCORING, refit=METRIC,
                          return_train_score=True, n_jobs=-1)
    search.fit(Xm_train, ym_train)

    table = pd.DataFrame(search.cv_results_)[[
        "params",
        f"mean_train_{METRIC}", f"mean_test_{METRIC}", f"std_test_{METRIC}",
        f"mean_test_{OTHER}",
    ]].sort_values(f"mean_test_{METRIC}", ascending=False)

    print(f"{name} best: {search.best_params_}")
    print(f"  cross-validated {METRIC}: {search.best_score_:.4f}")

    models[name + ", tuned"] = search.best_estimator_.named_steps["model"]
    return search, table.round(4)

In [ ]:
Cs = np.logspace(-4, 3, 8)
weights = [None, "balanced", {0: 1, 1: 10}, {0: 1, 1: 30}]
scalers = [StandardScaler(), RobustScaler(), MinMaxScaler()]

grid_linear = [
    # l2 + hinge only works with dual=True
    {"model__penalty": ["l2"], "model__loss": ["hinge"], "model__dual": [True],
     "model__C": Cs, "model__class_weight": weights, "prep__num__scale": scalers},
    # l2 + squared_hinge
    {"model__penalty": ["l2"], "model__loss": ["squared_hinge"], "model__dual": [False],
     "model__C": Cs, "model__class_weight": weights, "prep__num__scale": scalers},
    # l1 only works with squared_hinge and dual=False
    {"model__penalty": ["l1"], "model__loss": ["squared_hinge"], "model__dual": [False],
     "model__C": Cs, "model__class_weight": weights, "prep__num__scale": scalers},
]

search_lin, table_lin = tune("LinearSVC", grid_linear)
table_lin.head(10)

LinearSVC best: {'model__C': np.float64(0.001), 'model__class_weight': 'balanced', 'model__dual': False, 'model__loss': 'squared_hinge', 'model__penalty': 'l1', 'prep__num__scale': MinMaxScaler()}
  cross-validated recall: 1.0000


,params,mean_train_recall,mean_test_recall,std_test_recall,mean_test_roc_auc
173,"{'model__C': 0.001, 'model__class_weight': 'ba...",1.0000,1.0000,0.0000,0.8432
179,"{'model__C': 0.001, 'model__class_weight': {0:...",0.9925,0.9850,0.0200,0.7998
11,"{'model__C': 0.001, 'model__class_weight': {0:...",0.9762,0.9699,0.0367,0.7833
172,"{'model__C': 0.001, 'model__class_weight': 'ba...",0.9648,0.9597,0.0340,0.8436
10,"{'model__C': 0.001, 'model__class_weight': {0:...",0.9648,0.9550,0.0400,0.8395
177,"{'model__C': 0.001, 'model__class_weight': {0:...",0.9535,0.9549,0.0484,0.8474
178,"{'model__C': 0.001, 'model__class_weight': {0:...",0.9648,0.9549,0.0484,0.8476
171,"{'model__C': 0.001, 'model__class_weight': 'ba...",0.9585,0.9547,0.0400,0.8433
5,"{'model__C': 0.001, 'model__class_weight': 'ba...",0.9510,0.9499,0.0386,0.7583
58,"{'model__C': 10.0, 'model__class_weight': {0: ...",0.9435,0.9346,0.0516,0.8372


In [ ]:
grid_rbf = {
    "model__C": np.logspace(-3, 3, 7),
    "model__gamma": ["scale", 0.001, 0.01, 0.1, 1],
    "model__class_weight": [None, "balanced"],
}

search_rbf, table_rbf = tune("SVC rbf", grid_rbf)
table_rbf.head(10)

SVC rbf best: {'model__C': np.float64(0.01), 'model__class_weight': 'balanced', 'model__gamma': 'scale'}
  cross-validated recall: 0.8797


,params,mean_train_recall,mean_test_recall,std_test_recall,mean_test_roc_auc
5,"{'model__C': 0.01, 'model__class_weight': 'bal...",0.8869,0.8797,0.0746,0.8331
17,"{'model__C': 0.1, 'model__class_weight': 'bala...",0.8719,0.8745,0.0522,0.8459
8,"{'model__C': 0.01, 'model__class_weight': 'bal...",0.8781,0.8697,0.0675,0.8338
26,"{'model__C': 1.0, 'model__class_weight': 'bala...",0.8719,0.8695,0.0618,0.8471
15,"{'model__C': 0.1, 'model__class_weight': 'bala...",0.8605,0.8395,0.0599,0.8338
18,"{'model__C': 0.1, 'model__class_weight': 'bala...",0.8593,0.8395,0.0713,0.8350
27,"{'model__C': 1.0, 'model__class_weight': 'bala...",0.8455,0.8294,0.0911,0.8439
36,"{'model__C': 10.0, 'model__class_weight': 'bal...",0.8430,0.8244,0.1034,0.8453
46,"{'model__C': 100.0, 'model__class_weight': 'ba...",0.8354,0.8044,0.1037,0.8425
37,"{'model__C': 10.0, 'model__class_weight': 'bal...",0.8543,0.7894,0.0977,0.8319


In [7]:
grid_poly = {
    "model__degree": [2, 3, 4],
    "model__coef0": [0, 1],
    "model__C": [0.01, 0.1, 1, 10],          # large C with poly can take a very long time
    "model__gamma": ["scale", 0.01, 0.1],
    "model__class_weight": ["balanced"],
}

search_poly, table_poly = tune("SVC poly", grid_poly)
table_poly.head(10)

SVC poly best: {'model__C': 0.1, 'model__class_weight': 'balanced', 'model__coef0': 1, 'model__degree': 2, 'model__gamma': 0.01}
  cross-validated recall: 0.8695


,params,mean_train_recall,mean_test_recall,std_test_recall,mean_test_roc_auc
28,"{'model__C': 0.1, 'model__class_weight': 'bala...",0.8719,0.8695,0.0618,0.8467
31,"{'model__C': 0.1, 'model__class_weight': 'bala...",0.8656,0.8545,0.0911,0.8459
11,"{'model__C': 0.01, 'model__class_weight': 'bal...",0.8631,0.8545,0.0810,0.8450
9,"{'model__C': 0.01, 'model__class_weight': 'bal...",0.8593,0.8495,0.0906,0.8450
34,"{'model__C': 0.1, 'model__class_weight': 'bala...",0.8593,0.8495,0.0906,0.8450
46,"{'model__C': 1, 'model__class_weight': 'balanc...",0.8417,0.8195,0.1001,0.8453
16,"{'model__C': 0.01, 'model__class_weight': 'bal...",0.8304,0.8194,0.0615,0.8391
14,"{'model__C': 0.01, 'model__class_weight': 'bal...",0.8417,0.8144,0.0747,0.8425
49,"{'model__C': 1, 'model__class_weight': 'balanc...",0.8404,0.8094,0.1064,0.8429
22,"{'model__C': 0.1, 'model__class_weight': 'bala...",0.8000,0.8000,0.4000,0.8196


In [8]:
for name, s in [("LinearSVC", search_lin), ("SVC rbf", search_rbf), ("SVC poly", search_poly)]:
    best_C = s.best_params_["model__C"]
    grid_Cs = sorted({p["model__C"] for p in s.cv_results_["params"]})
    if best_C in (grid_Cs[0], grid_Cs[-1]):
        print(f"{name}: best C={best_C} is at the edge of the grid; widen the range")

LinearSVC: best C=0.001 is at the edge of the grid; widen the range
SVC rbf: best C=0.01 is at the edge of the grid; widen the range


In [9]:
results = compare(models, Xm_train, ym_train, cv)
results

,roc_auc,roc_auc spread,recall,recall spread,training roc_auc,fit ms,score ms per 1000 rows
model,,,,,,,
SVC linear,0.843,0.032,0.804,0.094,0.854,270.290,85.893
LinearSVC,0.845,0.029,0.824,0.103,0.856,22.514,22.209
SVC rbf,0.704,0.036,0.176,0.064,0.990,199.353,238.230
SVC poly,0.808,0.029,0.643,0.081,0.891,200.549,91.604
"LinearSVC, tuned",0.843,0.030,0.955,0.040,0.845,18.990,20.771
"SVC rbf, tuned",0.833,0.028,0.880,0.075,0.849,342.303,418.702
"SVC poly, tuned",0.847,0.029,0.869,0.062,0.855,212.122,102.305
